# Spatial Site Suitability Analysis 🗺️

In this project, we are trying to find the best spot to open a new retail shop or facility. 

First, we download a map of our target study area and overlay a precise grid of tiny squares (50-meter resolution). Next, we evaluate every single square based on three core spatial criteria:
1. **Road Proximity:** Being closer to roads is good for accessibility.
2. **Residential Density:** Being closer to homes ensures a better customer base.
3. **Competitor Avoidance:** Being farther from existing shops prevents market saturation.

Finally, we calculate a weighted Multi-Criteria Decision Analysis (MCDA) score, color-code the best locations in green and the worst in red, and generate an automated PDF report with our findings.

### Step 1: Installs and Imports

In [ ]:
# Install required packages
!pip install osmnx geopandas fpdf2 mapclassify -q

# Import all the tools we need
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
from shapely.geometry import box
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from google.colab import files

### Step 2: Get the map and draw the tiny squares

In [ ]:
place = "Sonadanga, Khulna, Bangladesh"

# Get the boundary of the area
boundary_gdf = ox.geocode_to_gdf(place)
minx, miny, maxx, maxy = boundary_gdf.total_bounds

# Set the size of our squares (about 50 meters)
cells = 0.0005 

# Draw the grid of squares
grid_cells = []
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        grid_cells.append(box(x, y, x + cells, y + cells))
        y += cells
    x += cells

grid_gdf = gpd.GeoDataFrame(geometry=grid_cells)
grid_gdf.set_crs(boundary_gdf.crs, inplace=True)

# Keep only the squares that are inside our town boundary
grid_gdf = grid_gdf[grid_gdf.intersects(boundary_gdf.union_all())]

# Find the center dot of every square and change to meter scale for measuring
grid_gdf['centroid'] = grid_gdf.geometry.centroid
grid_centroids_meter = gpd.GeoDataFrame(geometry=grid_gdf['centroid'], crs=grid_gdf.crs).to_crs(epsg=3857)

### Step 3: Find the roads and score them (Closer is better)

In [ ]:
# Get the roads
roads_gr = ox.graph_from_place(place, network_type='drive')
roads_gdf = ox.graph_to_gdfs(roads_gr, nodes=False)
roads_meter = roads_gdf.to_crs(epsg=3857)

# Measure distance from squares to the nearest road
grid_with_dist = gpd.sjoin_nearest(
    left_df=grid_centroids_meter,
    right_df=roads_meter,
    how="left",
    distance_col="dist_to_road_meters"
)
grid_with_dist = grid_with_dist[~grid_with_dist.index.duplicated(keep='first')]
grid_gdf["dist_to_road"] = grid_with_dist["dist_to_road_meters"]

# Give a score (1 is best, 0 is worst)
max_dist = grid_gdf["dist_to_road"].max()
grid_gdf["road_score"] = 1 - (grid_gdf["dist_to_road"] / max_dist)

### Step 4: Find other shops and score them (Farther is better)

In [ ]:
# Get the shops
shops = ox.features_from_place(place, tags={"shop": True})
shops_points = shops.geometry.centroid
shops_gdf = gpd.GeoDataFrame(geometry=shops_points, crs=shops.crs)
shops_meter = shops_gdf.to_crs(epsg=3857)

# Measure distance
grid_with_shop_dist = gpd.sjoin_nearest(
    left_df=grid_centroids_meter,
    right_df=shops_meter,
    how="left",
    distance_col="dist_to_shop_meters"
)
grid_with_shop_dist = grid_with_shop_dist[~grid_with_shop_dist.index.duplicated(keep='first')]
grid_gdf["dist_to_shop"] = grid_with_shop_dist["dist_to_shop_meters"]

# Give a score (Here, bigger distance = bigger score)
max_shop_dist = grid_gdf["dist_to_shop"].max()
grid_gdf["competitor_score"] = grid_gdf["dist_to_shop"] / max_shop_dist

### Step 5: Find residential homes and score them (Closer is better)

In [ ]:
# Get the homes
residential_buildings = ox.features_from_place(place, tags={"building": "residential"})
res_points = residential_buildings.geometry.centroid
res_gdf = gpd.GeoDataFrame(geometry=res_points, crs=residential_buildings.crs)
res_meter = res_gdf.to_crs(epsg=3857)

# Measure distance
grid_with_res_dist = gpd.sjoin_nearest(
    left_df=grid_centroids_meter,
    right_df=res_meter,
    how="left",
    distance_col="dist_to_res_meters"
)
grid_with_res_dist = grid_with_res_dist[~grid_with_res_dist.index.duplicated(keep='first')]
grid_gdf["dist_to_res"] = grid_with_res_dist["dist_to_res_meters"]

# Give a score
max_res_dist = grid_gdf["dist_to_res"].max()
grid_gdf["res_score"] = 1 - (grid_gdf["dist_to_res"] / max_res_dist)

### Step 6: Mix the scores together (Weighted Overlay)

In [ ]:
# Set how important each rule is
weights = {
    "road_score": 0.4,
    "res_score": 0.3,
    "competitor_score": 0.3
}

# Mix them up to get the final score
grid_gdf["final_score"] = (
    grid_gdf["road_score"] * weights["road_score"] +
    grid_gdf["res_score"] * weights["res_score"] +
    grid_gdf["competitor_score"] * weights["competitor_score"]
)

# Put them in groups (Low, Medium, High)
grid_gdf["suitability_class"] = pd.cut(
    x=grid_gdf["final_score"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

### Step 7: Draw and save static maps

In [ ]:
# Save the 3 small rule maps side-by-side
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(22, 7), facecolor='#F4F1EA')
legend_settings = {'orientation': "vertical", 'shrink': 0.7, 'pad': 0.02}

# Map 1: Roads
axes[0].set_facecolor('#F4F1EA')
grid_gdf.plot(column="road_score", cmap="RdYlGn", legend=True, ax=axes[0], edgecolor="none", alpha=0.9, legend_kwds=legend_settings)
roads_gdf.to_crs(grid_gdf.crs).plot(ax=axes[0], color="#2C3E50", linewidth=0.3, alpha=0.6, zorder=2)
axes[0].set_title("1. Road Proximity Score", fontsize=14, fontweight='bold', color='#2C3E50', pad=15)

# Map 2: Homes
axes[1].set_facecolor('#F4F1EA')
grid_gdf.plot(column="res_score", cmap="RdYlGn", legend=True, ax=axes[1], edgecolor="none", alpha=0.9, legend_kwds=legend_settings)
roads_gdf.to_crs(grid_gdf.crs).plot(ax=axes[1], color="#2C3E50", linewidth=0.3, alpha=0.6, zorder=2)
axes[1].set_title("2. Residential Proximity Score", fontsize=14, fontweight='bold', color='#2C3E50', pad=15)

# Map 3: Shops
axes[2].set_facecolor('#F4F1EA')
grid_gdf.plot(column="competitor_score", cmap="RdYlGn", legend=True, ax=axes[2], edgecolor="none", alpha=0.9, legend_kwds=legend_settings)
roads_gdf.to_crs(grid_gdf.crs).plot(ax=axes[2], color="#2C3E50", linewidth=0.3, alpha=0.6, zorder=2)
axes[2].set_title("3. Competitor Avoidance Score", fontsize=14, fontweight='bold', color='#2C3E50', pad=15)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine_position in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine_position].set_visible(True)
        ax.spines[spine_position].set_linewidth(2)
        ax.spines[spine_position].set_color('#2C3E50')

plt.tight_layout()
plt.savefig("three_criteria_breakdown.png", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()

# Save the big final map
fig, ax = plt.subplots(figsize=(12, 12), facecolor='#F4F1EA')
ax.set_facecolor('#F4F1EA')

grid_gdf.plot(
    column="final_score", cmap="YlOrRd", legend=True, ax=ax, edgecolor="none", alpha=0.9, zorder=1,
    legend_kwds={'label': "Final Suitability Score", 'orientation': "vertical", 'shrink': 0.75, 'pad': 0.02}
)
roads_gdf.to_crs(grid_gdf.crs).plot(ax=ax, color="#2C3E50", linewidth=0.5, alpha=0.7, zorder=2)

plt.title("Final Site Suitability Map - Sonadanga", fontsize=18, fontweight='bold', color='#2C3E50', loc='center', pad=20)
ax.set_xlabel("Longitude", fontsize=12, fontweight='bold', labelpad=10, color='#2C3E50')
ax.set_ylabel("Latitude", fontsize=12, fontweight='bold', labelpad=10, color='#2C3E50')

for spine_position in ['top', 'bottom', 'left', 'right']:
    ax.spines[spine_position].set_visible(True)
    ax.spines[spine_position].set_linewidth(2.5)
    ax.spines[spine_position].set_color('#2C3E50')

ax.tick_params(axis='both', which='major', labelsize=10, colors='#2C3E50')
plt.tight_layout()
plt.savefig("final_suitability_map.png", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()

### Step 8: Calculate Summary Statistics

In [ ]:
# Count how many squares are in each group
summary_stats = grid_gdf.groupby("suitability_class", observed=False).size().reset_index(name="total_cells")
total_cells_count = summary_stats["total_cells"].sum()
summary_stats["percentage"] = (summary_stats["total_cells"] / total_cells_count) * 100

# Make it look nice for the report
summary_stats["percentage"] = summary_stats["percentage"].round(2).astype(str) + " %"
summary_stats.rename(columns={
    "suitability_class": "Suitability Level",
    "total_cells": "Cell Count (50m)",
    "percentage": "Percentage (%)"
}, inplace=True)
summary_stats

### Step 9: Interactive Web Map (Folium)

In [ ]:
# Clean up the data for the web map
grid_folium = grid_gdf.drop(columns=['centroid'], errors='ignore').to_crs(epsg=4326)
centroid_point = grid_folium.geometry.union_all().centroid

# Make the web map
m = folium.Map(location=[centroid_point.y, centroid_point.x], zoom_start=14, tiles="OpenStreetMap")
colormap = cm.LinearColormap(colors=['red', 'yellow', 'green'], vmin=0, vmax=1)

def my_style_function(feature):
    score = feature['properties']['final_score']
    return {
        'fillColor': colormap(score),
        'color': 'black',
        'weight': 0.3,
        'fillOpacity': 0.6
    }

folium.GeoJson(
    data=grid_folium,
    style_function=my_style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['final_score', 'suitability_class'],
        aliases=['Suitability Score:', 'Class:']
    )
).add_to(m)

# Display the map
m

### Step 10: Generate PDF Report & Download Data

In [ ]:
class PDFReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.set_text_color(40, 40, 40)
        self.cell(0, 10, "Simple Suitability Analysis Report", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
        self.line(10, 20, 200, 20)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f"Page {self.page_no()}", align="C")

pdf = PDFReport(orientation="P", unit="mm", format="A4")
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Project Overview
pdf.set_font("Helvetica", "B", 16)
pdf.set_text_color(0, 51, 102)
pdf.cell(0, 10, "1. Project Overview", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_font("Helvetica", "", 11)
pdf.set_text_color(0, 0, 0)
pdf.multi_cell(0, 6, "The primary goal is to find the most suitable 50-meter land parcels for establishing a new retail business by evaluating geographic conditions.")
pdf.ln(5)

# Location
pdf.set_font("Helvetica", "B", 12)
pdf.cell(0, 8, "Location & Objective:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_font("Helvetica", "", 11)
pdf.multi_cell(0, 6, "Study Area: Sonadanga, Khulna, Bangladesh. Our objective is to pinpoint exact blocks where a business can maximize foot traffic while minimizing market competition.")
pdf.ln(5)

# Criteria
pdf.set_font("Helvetica", "B", 16)
pdf.set_text_color(0, 51, 102)
pdf.cell(0, 10, "2. Analysis Criteria & Justification", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_text_color(0, 0, 0)
pdf.set_font("Helvetica", "", 11)
pdf.multi_cell(0, 6, "A. Road Proximity (40%): Closer to roads is better for customers.\nB. Residential Density (30%): Closer to homes is better for sales.\nC. Competitor Avoidance (30%): Farther from other shops is better.")
pdf.ln(5)

# Limitations
pdf.set_font("Helvetica", "B", 16)
pdf.set_text_color(0, 51, 102)
pdf.cell(0, 10, "3. Data Source & Limitations", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_text_color(0, 0, 0)
pdf.set_font("Helvetica", "", 11)
pdf.multi_cell(0, 6, "All data came from OpenStreetMap. Sometimes this map is missing new homes or small shops, so it might not be 100% perfect, but it is great for learning.")
pdf.ln(5)

# Add Maps
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.set_text_color(0, 51, 102)
pdf.cell(0, 10, "4. Individual Criteria Analysis", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.image("three_criteria_breakdown.png", x=10, w=190)
pdf.ln(10)

pdf.add_page()
pdf.cell(0, 10, "5. Final Weighted Suitability Output", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.image("final_suitability_map.png", x=30, w=150)
pdf.ln(10)

# Add Table
pdf.set_font("Helvetica", "B", 14)
pdf.set_text_color(0, 0, 0)
pdf.cell(0, 10, "Summary Statistics", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_font("Helvetica", "B", 10)
pdf.cell(50, 8, "Suitability Class", border=1, align="C")
pdf.cell(50, 8, "Total 50m Cells", border=1, align="C")
pdf.cell(50, 8, "Percentage (%)", border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")

pdf.set_font("Helvetica", "", 10)
for index, row in summary_stats.iterrows():
    pdf.cell(50, 8, str(row['Suitability Level']), border=1, align="C")
    pdf.cell(50, 8, str(row['Cell Count (50m)']), border=1, align="C")
    pdf.cell(50, 8, str(row['Percentage (%)']), border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
pdf.ln(10)

# Author
pdf.set_font("Helvetica", "B", 12)
pdf.cell(0, 6, "Prepared By:", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_font("Helvetica", "B", 14)
pdf.set_text_color(0, 102, 51)
pdf.cell(0, 8, "MD MOKAMMEL MORSHED", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.set_font("Helvetica", "U", 11)
pdf.set_text_color(0, 0, 255)
pdf.cell(0, 6, "https://meheran.dev", new_x=XPos.LMARGIN, new_y=YPos.NEXT)

pdf.output("Simple_Suitability_Analysis_Report.pdf")

# Save Data
grid_gdf_to_save = grid_gdf.drop(columns=['centroid'], errors='ignore')
grid_gdf_to_save.to_file("final_grid_suitability.geojson", driver="GeoJSON")

# Download
files.download("Simple_Suitability_Analysis_Report.pdf")
files.download("final_grid_suitability.geojson")